# Finetuning to follow instructions

In [75]:
import numpy as np

import matplotlib.pyplot as plt

import tiktoken

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm

import tensorflow

import torchmetrics

## Preparing a dataset for supervised instruction finetuning

In [27]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [28]:
data[999]

{'instruction': "What is an antonym of 'complicated'?",
 'input': '',
 'output': "An antonym of 'complicated' is 'simple'."}

In [29]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry["instruction"]}"
    )

    input_text = f"\n\n### Input:\n{entry["input"]}" if entry["input"] else ""

    return instruction_text + input_text

In [30]:
print(format_input(data[999]))

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?


In [31]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]["output"]}"

print(model_input + desired_response)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [32]:
train_split = int(len(data) * 0.85)
test_split = int(len(data) * 0.1)
val_split = len(data) - train_split - test_split

train_data = data[:train_split]
test_data = data[train_split:train_split + test_split]
val_data = data[train_split + test_split:]

In [33]:
print("Training set length:", len(train_data))
print("Val set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Val set length: 55
Test set length: 110


## Organizing data into training batches

In [34]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry["output"]}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, idx):
        return self.encoded_texts[idx]

    def __len__(self):
        return len(self.data)

tokenizer = tiktoken.get_encoding("gpt2")

In [35]:
def custom_collate_draft_1(batch,
                           pad_token_id=50256,
                           device=torch.device("cuda")):

    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst = []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [36]:
inputs1 = [0, 1, 2, 3, 4]
inputs2 = [5, 6]
inputs3 = [7, 8, 9]

batch = (
    inputs1,
    inputs2,
    inputs3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]], device='cuda:0')


In [37]:
def custom_collate_draft_2(batch,
                           pad_token_id=50256,
                           device=torch.device("cuda")):

    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [38]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]], device='cuda:0')
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]], device='cuda:0')


In [39]:
def custom_collate_draft_3(batch,
                           pad_token_id=50256,
                           ignore_idx=-100,
                           allowed_max_length=None,
                           device=torch.device("cuda")):

    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_idx

        if allowed_max_length:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [40]:
inputs, targets = custom_collate_draft_3(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]], device='cuda:0')
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]], device='cuda:0')


## Creating data loaders for an instruction dataset

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [67]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_draft_3,
    allowed_max_length=1024
)

torch.manual_seed(123)
num_workers = 0
batch_size = 8

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

## Loading a pretrained LLM

In [43]:
from load_gpt2 import load_gpt2
model, CONFIG = load_gpt2("medium")
model.to(device)
print(model, CONFIG)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 8503.94it/s]


GPT2Model(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (ke

In [44]:
from train_loop import train, test_model, generate_text_test

In [45]:
for param in model.parameters():
    param.requires_grad = False
for param in model.linear_out.parameters():
    param.requires_grad = True
for param in model.transformer_blocks[-1].parameters():
    param.requires_grad = True

EPOCHS = 10
LR = 0.0004
STEP_SIZE = 3
GAMMA = 0.1

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, STEP_SIZE, GAMMA)

In [46]:
model, train_losses, val_losses, epochs = train(model,
                                                loss_fn,
                                                optimizer,
                                                scheduler,
                                                train_loader,
                                                val_loader,
                                                test_loader,
                                                device,
                                                EPOCHS)


Epoch: 1 | Train loss: 1.1002753140597508
          Val loss: 0.8327871901648385

Epoch: 2 | Train loss: 0.5866606834119764
          Val loss: 0.7942857827459063

Epoch: 3 | Train loss: 0.409897829438078
          Val loss: 0.8223178301538739

Epoch: 4 | Train loss: 0.27574630852403315
          Val loss: 0.8555823564529419

Epoch: 5 | Train loss: 0.23481968096617994
          Val loss: 0.8921430110931396

Epoch: 6 | Train loss: 0.21192411436089154
          Val loss: 0.94187102999006

Epoch: 7 | Train loss: 0.18928761574728736
          Val loss: 0.9538241028785706

Epoch: 8 | Train loss: 0.18654443798907872
          Val loss: 0.9628294961793082

Epoch: 9 | Train loss: 0.18498242312464222
          Val loss: 0.9698273369244167

Epoch: 10 | Train loss: 0.18243887224074068
          Val loss: 0.9706755365644183
          Final results: Test loss: 1.1091909749167306


In [47]:
torch.save(model.state_dict(), "model.pth")

In [64]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)
text = generate_text_test(35, input_text, model, device, 1.3, 20, 50256)

print(text)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:
The chef prepares the meal by adding salt.


In [63]:
stripped_text = text[len(input_text):].strip("### Response:\n")

stripped_text

'The chef (the chef) explains that he cooked the meal every day.'

## Extracting and saving responses

In [74]:
torch.manual_seed(123)

for entry in test_data[:3]:

    input_text = format_input(entry)

    generated_text = generate_text_test(256, input_text, model, device, 1.4, 25, 50256)

    response_text = generated_text[len(input_text):].replace("### Response:\n", "").strip()

    print(input_text)
    print(f"\nCorrect response:\n>> {entry["output"]}")
    print(f"\nModel response:\n>> {response_text}")
    print("------------------------------------------------")

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a cheetah.
------------------------------------------------
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of clouds a region typically has is typically precipitation and/or snow.
------------------------------------------------
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.

Model response

In [76]:
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)

    generated_text = generate_text_test(256, input_text, model, device, 1.4, 25, 50256)

    response_text = generated_text[len(input_text):].replace("### Response:\n", "").strip()

    test_data[i]["model_response"] = response_text

with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)

100%|██████████| 110/110 [01:16<00:00,  1.45it/s]
